## 1. Geração da Tabela de Estatísticas descritivas dos perfis de participação (Rota B - Moodle), a partir do saída "matriz_rota_b_clusterizada.csv" do notebook "05_modelagem_rota_B_VF_artigo.ipynb".

In [1]:
import pandas as pd
import numpy as np
import os
import sys # Importado para garantir a interrupção correta em caso de erro

# --- Configurações Globais ---
from pathlib import Path
# Localiza automaticamente os arquivos gerados pelas etapas 05 e 06
candidatos_rota_b = [
    Path('dados_output/matriz_rota_b_clusterizada.csv'),
    Path('matriz_rota_b_clusterizada.csv'),
    Path('/content/dados_output/matriz_rota_b_clusterizada.csv'),
    Path('/content/matriz_rota_b_clusterizada.csv')
]
file_path_rota_b = next((str(p) for p in candidatos_rota_b if p.exists()), 'matriz_rota_b_clusterizada.csv')

candidatos_rota_a = [
    Path('dados_output/matriz_rota_a_clusterizada_final.csv'),
    Path('matriz_rota_a_clusterizada_final.csv'),
    Path('/content/dados_output/matriz_rota_a_clusterizada_final.csv'),
    Path('/content/matriz_rota_a_clusterizada_final.csv')
]
file_path_rota_a = next((str(p) for p in candidatos_rota_a if p.exists()), 'matriz_rota_a_clusterizada_final.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"Iniciando geração da tabela LaTeX para a Rota B (Perspectiva de Medianas e Notas)...")
print("-" * 70)

# 1. Carregar o arquivo CSV da Rota B
file_name_rota_b = 'matriz_rota_b_clusterizada.csv'
# file_path_rota_b localizado nos candidatos acima

try:
    # AJUSTE AQUI: Adicionado sep=';', decimal=',' e encoding='utf-8'
    df_rota_b = pd.read_csv(file_path_rota_b, sep=';', decimal=',', encoding='utf-8')
    print(f"Arquivo '{file_name_rota_b}' carregado com sucesso.")
except FileNotFoundError:
    print(f"[ERRO CRÍTICO] O arquivo '{file_name_rota_b}' não foi encontrado em '{file_path_rota_b}'.")
    sys.exit()
except Exception as e:
    print(f"Ocorreu um erro inesperado ao carregar o arquivo CSV: {e}")
    print("Dica: Se o erro persistir, o arquivo pode estar usando ',' como separador, mas com aspas faltando em textos.")
    sys.exit()

# 2. Verificar e exibir as colunas disponíveis (para depuração)
print("\nColunas disponíveis no DataFrame da Rota B (carregado do CSV):")
print(df_rota_b.columns.tolist())
print("-" * 70)

# 3. Preparar os dados para a tabela LaTeX
required_cols = [
    'Persona_Educacional',
    'Cluster_K2',
    'id_inscricoes',
    'nota_final_kdd',
    'progresso_kdd',
    'Horas_Reais_Engajamento',
    'qtd_tentativas'
]

if not all(col in df_rota_b.columns for col in required_cols):
    print("[ERRO] Algumas colunas necessárias não foram encontradas no DataFrame da Rota B.")
    missing = [col for col in required_cols if col not in df_rota_b.columns]
    print(f"Colunas ausentes: {missing}")
    sys.exit()

total_inscricoes_rota_b = len(df_rota_b)

# Como adicionamos decimal=',', o Pandas já deve ter convertido as numéricas para float.
# Caso alguma coluna numérica ainda esteja como string, forçamos a conversão:
cols_numericas = ['nota_final_kdd', 'progresso_kdd', 'Horas_Reais_Engajamento', 'qtd_tentativas']
for col in cols_numericas:
    df_rota_b[col] = pd.to_numeric(df_rota_b[col], errors='coerce')

# Agrupamento e cálculo das estatísticas (Medianas)
df_agrupado_rota_b = df_rota_b.groupby(['Cluster_K2', 'Persona_Educacional']).agg(
    N=('id_inscricoes', 'count'),
    Nota_Mediana=('nota_final_kdd', 'median'),
    Nota_Std=('nota_final_kdd', 'std'),
    Progresso_Mediano=('progresso_kdd', 'median'),
    Progresso_Std=('progresso_kdd', 'std'),
    Horas_Medianas=('Horas_Reais_Engajamento', 'median'),
    Horas_Std=('Horas_Reais_Engajamento', 'std'),
    Tentativas_Medianas=('qtd_tentativas', 'median'),
    Tentativas_Std=('qtd_tentativas', 'std')
).reset_index()

# Calcular o percentual de cada cluster
df_agrupado_rota_b['Percentual'] = (df_agrupado_rota_b['N'] / total_inscricoes_rota_b) * 100

# Ordenar pelo número do cluster
df_agrupado_rota_b = df_agrupado_rota_b.sort_values(by='Cluster_K2')

# Função auxiliar para formatar números no padrão brasileiro (1.000,00)
def format_br(val, is_int=False):
    if pd.isna(val):
        return "-"
    if is_int:
        return f"{int(val):_}".replace('_', '.')
    return f"{val:.1f}".replace('.', ',')

# 4. Gerar a string LaTeX com a estrutura apropriada
latex_str_rota_b = r"""\begin{table}[!htbp]
    \centering
    % Redução máxima recomendada para tabelas de dados
    \scriptsize

    % Reduz o espaço em branco horizontal entre as colunas ao mínimo aceitável
    \setlength{\tabcolsep}{2pt}

    % Aumenta o espaçamento vertical entre as linhas
    \renewcommand{\arraystretch}{1.25}

    \caption{Estatísticas descritivas dos perfis de participação (Rota B - Moodle)}
    \label{tab:descritiva_rota_b_medianas}

    % Coluna de texto reduzida para 3.5cm
    \begin{tabular}{@{} l p{3.5cm} c c c c c @{}}
        \toprule
        \textbf{Cluster} & \textbf{Perfil (Taxonomia)} & \textbf{N (\%)} &
        % Truque nativo para quebrar os cabeçalhos longos em duas linhas
        \textbf{\begin{tabular}{@{}c@{}}Nota \\ Final\end{tabular}} &
        \textbf{\begin{tabular}{@{}c@{}}Progresso \\ (\%)\end{tabular}} &
        \textbf{\begin{tabular}{@{}c@{}}Horas \\ Engaj.\end{tabular}} &
        \textbf{Tentativas} \\
        \midrule
"""

# Iterar sobre as linhas do DataFrame agrupado para preencher a tabela
for index, row in df_agrupado_rota_b.iterrows():
    cluster_label = f"C{int(row['Cluster_K2'])}"
    persona = row['Persona_Educacional']

    n_formatado = format_br(row['N'], is_int=True)
    pct_formatado = format_br(row['Percentual'])
    n_pct = f"{n_formatado} ({pct_formatado}\\%)"

    nota = f"{format_br(row['Nota_Mediana'])} $\\pm$ {format_br(row['Nota_Std'])}"
    prog = f"{format_br(row['Progresso_Mediano'])} $\\pm$ {format_br(row['Progresso_Std'])}"
    horas = f"{format_br(row['Horas_Medianas'])} $\\pm$ {format_br(row['Horas_Std'])}"
    tent = f"{format_br(row['Tentativas_Medianas'])} $\\pm$ {format_br(row['Tentativas_Std'])}"

    latex_str_rota_b += f"        {cluster_label} & {persona} & {n_pct} & {nota} & {prog} & {horas} & {tent} \\\\\n"

# Fechamento da tabela e inclusão da Fonte e Nota
latex_str_rota_b += r"""        \bottomrule
    \end{tabular}

    \vspace{0.15cm}

    % \textwidth garante que a nota acompanhará o limite exato do documento
    \parbox{\textwidth}{
        \raggedright \scriptsize
        \textbf{Fonte:} Elaborado pelos autores (2026). \par
        \vspace{0.1cm}
        \textbf{Nota:} As medianas evidenciam a sustentação empírica dos rótulos, demonstrando a correlação entre o sucesso acadêmico e o maior investimento de tempo na plataforma. \par
    }
\end{table}"""

print(latex_str_rota_b)

Iniciando geração da tabela LaTeX para a Rota B (Perspectiva de Medianas e Notas)...
----------------------------------------------------------------------
Arquivo 'matriz_rota_b_clusterizada.csv' carregado com sucesso.

Colunas disponíveis no DataFrame da Rota B (carregado do CSV):
['Persona_Educacional', 'Cluster_K2', 'PC1', 'PC2', 'PC3', 'id_inscricoes', 'Nome_Breve', 'Nome_Formacao', 'carga_horaria', 'datahora_inscricao', 'Sexo', 'data_nascimento', 'Municipio', 'Tipo_Instituicao', 'SRE', 'id_discente_anonimo', 'Cargo_imputado', 'Funcao_imputado', 'Tipo_Instituicao_imputado', 'Sexo_imputado', 'Escolaridade_imputado', 'teto_curso', 'carga_horaria_winsor', 'Idade', 'Idade_imputado', 'Macro_Cargo', 'Escolaridade_Ordinal', 'Codigo_IBGE', 'nome_curto_do_curso', 'Ano_Oferta', 'status_geral', 'situacao_cursista', 'qtd_tentativas', 'progresso_kdd', 'flg_nota_imputada', 'nota_final_kdd', 'flg_tempo_imputado', 'minutos_totais_log1p', 'dias_permanencia_log1p', 'dias_duracao_curso', 'primeiro_a

## 2. Geração da Tabela de Estatísticas descritivas dos perfis de participação (Rota A - Moodle), a partir do saída "matriz_rota_a_clusterizada.csv" do notebook "06_modelagem_rota_a_VF_artigo.ipynb".

In [2]:
import pandas as pd
import numpy as np

# 1. Carregar os dados
df = pd.read_csv(file_path_rota_a)

# 2. Agrupar pelo nome correto da coluna de cluster (Cluster_K4)
agrupado = df.groupby('Cluster_K4').agg(
    N=('id_inscricoes', 'count'),
    Nota_Mediana=('nota_final_kdd', 'median'),
    Nota_Std=('nota_final_kdd', 'std'),
    Progresso_Mediano=('progresso_kdd', 'median'),
    Progresso_Std=('progresso_kdd', 'std'),
    Horas_Medianas=('Horas_Reais_Engajamento', 'median'),
    Horas_Std=('Horas_Reais_Engajamento', 'std'),
    Tentativas_Medianas=('qtd_tentativas', 'median'),
    Tentativas_Std=('qtd_tentativas', 'std')
).reset_index()

# 3. Mapear os rótulos da taxonomia antiga
# ATENÇÃO: Verifique se a ordem (0, 1, 2, 3) corresponde corretamente aos perfis reais dos seus dados.
mapeamento_taxonomia = {
    0: 'Certificação Expressa',
    1: 'Evasão Crônica',
    2: 'Engajamento Orgânico',
    3: 'Resiliência com Fricção'
}
agrupado['Persona'] = agrupado['Cluster_K4'].map(mapeamento_taxonomia)

# 4. Calcular percentuais
total_n = agrupado['N'].sum()
agrupado['Percentual'] = (agrupado['N'] / total_n) * 100

# 5. Ordenar os clusters do 0 ao 3
agrupado = agrupado.sort_values('Cluster_K4')

# 6. Gerar o código LaTeX
latex_str = r"""\begin{table}[htbp]
\centering
\caption{Estatísticas descritivas dos perfis de participação (Rota A - Moodle) - Valores em Mediana}
\label{tab:descritiva_rota_a_medianas}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llccccc}
\toprule
\textbf{Cluster} & \textbf{Perfil (Taxonomia)} & \textbf{N (\%)} & \textbf{Nota Final} & \textbf{Progresso (\%)} & \textbf{Horas Engaj.} & \textbf{Tentativas} \\
\midrule
"""

for _, row in agrupado.iterrows():
    cluster_num = int(row['Cluster_K4'])
    persona = row['Persona']
    n_pct = f"{int(row['N']):,} ({row['Percentual']:.1f}\\%)".replace(',', '.')

    # Utilizando as novas variáveis de mediana
    nota = f"{row['Nota_Mediana']:.1f} $\\pm$ {row['Nota_Std']:.1f}"
    prog = f"{row['Progresso_Mediano']:.1f} $\\pm$ {row['Progresso_Std']:.1f}"
    horas = f"{row['Horas_Medianas']:.1f} $\\pm$ {row['Horas_Std']:.1f}"
    tent = f"{row['Tentativas_Medianas']:.1f} $\\pm$ {row['Tentativas_Std']:.1f}"

    # Corrigido para o & nativo do LaTeX
    latex_str += f"C{cluster_num} & {persona} & {n_pct} & {nota} & {prog} & {horas} & {tent} \\\\\n"

# Fechamento da tabela e inclusão da Fonte e Nota
latex_str += r"""\bottomrule
\end{tabular}%
}
\vspace{0.15cm}
{\raggedright \footnotesize \textbf{Fonte:} Elaborado pelos autores (2026). \par}
{\raggedright \footnotesize \textbf{Nota:} As medianas evidenciam a sustentação empírica dos rótulos, demonstrando a correlação entre o sucesso acadêmico e o maior investimento de tempo na plataforma. \par}
\end{table}"""

print(latex_str)

\begin{table}[htbp]
\centering
\caption{Estatísticas descritivas dos perfis de participação (Rota A - Moodle) - Valores em Mediana}
\label{tab:descritiva_rota_a_medianas}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llccccc}
\toprule
\textbf{Cluster} & \textbf{Perfil (Taxonomia)} & \textbf{N (\%)} & \textbf{Nota Final} & \textbf{Progresso (\%)} & \textbf{Horas Engaj.} & \textbf{Tentativas} \\
\midrule
C0 & Certificação Expressa & 86.775 (45.3\%) & 91.0 $\pm$ 13.4 & 100.0 $\pm$ 10.4 & 0.0 $\pm$ 1.7 & 1.0 $\pm$ 0.0 \\
C1 & Evasão Crônica & 73.175 (38.2\%) & 0.0 $\pm$ 7.9 & 0.0 $\pm$ 14.9 & 0.0 $\pm$ 1.0 & 1.0 $\pm$ 0.0 \\
C2 & Engajamento Orgânico & 27.977 (14.6\%) & 94.0 $\pm$ 33.4 & 100.0 $\pm$ 29.0 & 14.4 $\pm$ 10.8 & 1.0 $\pm$ 0.1 \\
C3 & Resiliência com Fricção & 3.485 (1.8\%) & 78.0 $\pm$ 36.1 & 87.5 $\pm$ 36.8 & 0.0 $\pm$ 4.6 & 4.0 $\pm$ 0.0 \\
\bottomrule
\end{tabular}%
}
\vspace{0.15cm}
{\raggedright \footnotesize \textbf{Fonte:} Elaborado pelos autores (2026). \par}
{\raggedrigh